In [1]:
import os
!pip install kaggle gradio

In [2]:
!kaggle datasets download -d dansbecker/hot-dog-not-hot-dog
!unzip -q hot-dog-not-hot-dog.zip -d .hot_dog_data

Dataset URL: https://www.kaggle.com/datasets/dansbecker/hot-dog-not-hot-dog
License(s): other
100% 89.3M/89.3M [00:00<00:00, 114MB/s]



In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_dir = '.hot_dog_data/train'
test_dir = '.hot_dog_data/test'
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
)

validation_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    test_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

print("Class index: ", train_generator.class_indices)

Found 498 images belonging to 2 classes.
Found 500 images belonging to 2 classes.
Class index:  {'hot_dog': 0, 'not_hot_dog': 1}


In [15]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,940,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=30,
    validation_data=validation_generator,
    validation_steps=len(validation_generator)
)

Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.5884 - loss: 0.6627 - val_accuracy: 0.5360 - val_loss: 0.7002
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.6245 - loss: 0.6623 - val_accuracy: 0.5520 - val_loss: 0.6867
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.6185 - loss: 0.6529 - val_accuracy: 0.5180 - val_loss: 0.7292
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.6064 - loss: 0.6525 - val_accuracy: 0.5340 - val_loss: 0.7098
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.6386 - loss: 0.6457 - val_accuracy: 0.5420 - val_loss: 0.6935
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.6265 - loss: 0.6467 - val_accuracy: 0.5400 - val_loss: 0.7086
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.6165 - loss: 0.6541 - val_accuracy: 0.5500 - val_loss: 0.6932
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.6466 - loss: 0.6470 - val_accuracy: 0.5460 - val_loss:

In [18]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image
def predict(img):
  img=img.resize((150,150))
  img_array = image.img_to_array(img)
  img_array = np.expand_dims(img_array,axis=0)/255.0
  prediction = model.predict(img_array)[0][0]
  is_hotdog_prob = 1 - prediction if 'hot_dog' == list(train_generator.class_indices.keys())[0] else prediction
  if is_hotdog_prob>0.5:
    return f"HOTDOG (Confidence - {is_hotdog_prob:.2%})"
  else:
    return f"NOT HOTDOG (Confidence - {1 - is_hotdog_prob:.2%})"
interface = gr.Interface(
    fn = predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Hotdog or Not Hotdog",
    description = "Upload an image to check"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f00aaeb2755cdf4bbd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
